你这份综合分析脚本（Q1-Q4）不仅代码逻辑严密，而且在量化研究的**深度**上已经触及了高频/日内策略最本质的痛点。

通过你提供的截图结果，我看到了一个非常震撼的现象，这在量化界被称为**“均值回归的陷阱”**或**“冲击衰减（Alpha Decay）”**。以下是我的深度解读和下一步行动建议：

---

### 1. 核心结果解读：你揭开了“虚假动量”的真面目

#### **A. 致命的“VWAP-Close 差值” (Q3 & Q4)**
这是你整份分析中最具价值的发现：
*   **现象**：在 D9（预测概率最高组）中，`VWAP` 离场收益是 **+3.28bps**，而 `Close` 离场收益是 **-13.54bps**。两者的差值（Gap）高达 **16.82bps**！
*   **物理意义**：你选出的 Top-N 股票在 10:35 之后确实在涨，但它们呈现出一种**“脉冲式冲高 -> 快速回落”**的模式。
    *   VWAP 离场抓住了整个 5 分钟 Bar 的平均价格，包含了一部分冲高时的“高价”。
    *   Close 离场抓住了 Bar 结束那一秒的价格，此时价格已经回落。
*   **结论**：你的模型目前学到的是**“日内动量的最后一次喷发”**。如果你不能通过算法交易（如分批成交）卖在均价，而是只能在收盘那一刻“市价砸出”，这个策略在扣除 15bps 成本后将面临巨大亏损。

#### **B. Q1/Q2 的画像：模型偏爱“放量过热股”**
*   观察 Top-50 的画像：`rel_vol` 为 **2.90**（全场最高），`X1_zscore` 为 **0.85**（明显高于均值）。
*   这意味着模型倾向于寻找**“正在放量且价格在均线上方”**的票。
*   **矛盾点**：这组票的 `excess` 是正的（+3.99），但 `close_ret` 是负的（-6.15）。
*   **深度解读**：模型虽然识别出了 Alpha（超额），但这些股票的 Beta 属性太强，波动率太大。在小盘股里，**放量往往意味着流动性溢价的枯竭点**。

#### **C. Q3 的 D7-D9 异常单调性**
*   你发现 `raw_ret` (VWAP) 在 D7 到 D9 是上升的，但 `close_ret` 却是剧烈下降的。
*   **结论**：预测概率（pred_prob）越高，**“冲高回落”的现象就越严重**。模型越有把握，说明特征越极端，而特征越极端，市场的非理性程度越高，回落就越狠。

---

### 2. 我的看法与意见

1.  **策略的生存空间取决于执行端**：
    目前的 V2-Regression 模型是一个典型的**“执行增强（Execution Alpha）”**信号。如果你是一家券商的交易部，要替客户减持，这个信号能帮你找到最佳减持点；但作为纯主动投资，它的“净利润”被回落过程（VWAP-Close Gap）吃掉了。

2.  **回归模型比分类模型更“诚实”**：
    Q2 的分析证明了之前的二分类标签确实存在严重的“过拟合高波动”倾向。回归模型虽然在 BPS 上看起来低，但它揭示了真实的路径质量。

3.  **15bps 成本是目前的“生死线”**：
    由于 `VWAP-Close 差值`（~16bps）已经和你的交易成本（15bps）持平，意味着**如果你卖不准，你就是在亏损**。

---

### 3. 下一步行动计划：转向“路径质量（Path Quality）”

我们需要让模型去学习：**谁的上涨是“稳健的”，谁的上涨是“脉冲的”？**

#### **任务 1：引入“稳定性”特征（Meta-Filtering）**
我们需要训练一个“二阶模型”或者增加特征，专门预测 `VWAP - Close` 的差值。
*   **动作**：加入特征 `X1_std_last_30m`（过去 30 分钟价格的稳定性）。如果波动太剧烈，即使预测涨幅大也不做。

#### **任务 2：路径质量过滤 (ER)**
正如之前建议的，引入 **Efficiency Ratio (ER)**。
*   逻辑：如果过去 1 小时的上涨是靠 1-2 个大 Bar 拉起来的（ER 低），这种票大概率会回落。我们只要那些“稳步推升”的票。

#### **任务 3：止盈策略回测**
不要死守 120 分钟。如果 60 分钟内已经达到了 `pred_ret` 的目标，提前出场。

---

### 4. 建议执行代码：路径稳定性特征增强

请在特征生成环节加入以下代码，来区分“稳健动量”和“脉冲诱多”：

```python
"""
S10: 路径稳定性与脉冲检测特征
目标：通过量化上涨的"顺滑度"，过滤掉那些容易"冲高回落"的样本
"""

def add_path_stability_features(df):
    logger.info("构建路径稳定性特征...")
    df = df.sort_values(["SecuCode", "date", "entry_time"])
    grp = df.groupby(["SecuCode", "date"])
    
    # 1. 效率系数 (ER): 位移 / 路程
    # 逻辑：ER接近1表示直线拉升，ER接近0表示剧烈震荡
    df["abs_ret_5m"] = grp["close"].pct_change().abs()
    df["total_path"] = grp["abs_ret_5m"].rolling(12).sum() # 过去1小时
    df["net_displacement"] = (df["close"] / grp["close"].shift(12) - 1).abs()
    df["efficiency_ratio"] = df["net_displacement"] / df["total_path"]

    # 2. 脉冲强度 (Impulse): 当前5min涨幅 / 过去1小时最大涨幅
    df["max_ret_1h"] = grp["abs_ret_5m"].rolling(12).max()
    df["impulse_factor"] = df["abs_ret_5m"] / df["max_ret_1h"]

    # 3. 价格相对于VWAP的稳定性 (Z-Score的波动)
    df["X1_volatility"] = grp["X1_zscore"].rolling(6).std() # 过去30分钟X1的抖动

    # 4. 预测目标改进 (核心建议): 
    # 预测不再是简单的 excess_ret，而是 (VWAP + Close) / 2 的超额收益
    # 迫使模型寻找在Bar结束时依然能站稳的股票
    df["mid_exit_ret"] = (df["raw_ret"] + df["close_ret"]) / 2.0
    
    return df
```

### 5. 总结

你现在的分析已经让你站在了**“收割脉冲”还是“跟随趋势”**的分水岭上。

**老板可能会问你：** “为什么模型分越高，收盘价反而跌得越厉害？”
**你可以回答：** “因为高分股是市场情绪的极值点。我们目前的特征（X1/X2/RelVol）捕捉的是**‘交易热度’**。在 T+0 环境下，热度过快积聚会导致收盘前的获利回吐（Mean Reversion）。下一步我们将引入**路径效率系数**，剔除脉冲个股，锁定稳健趋势。”

**你需要我帮你针对这个“VWAP-Close Gap”设计一个专门的“防回落”过滤模型吗？**

你这版对比实验做得**极其到位**。通过将训练标签和回测离场价同时切换到 `Close`，你实际上完成了量化研究中最严苛的**“脱水测试”**。

### 一、 核心结果解读：策略的“成色”分析

从你的三张表格中，我们可以读出几个具有实战指导意义的结论：

#### 1. 信号的“脉冲性”得到了量化（表1 差值分析）
*   **现象**：在全换手场景下，策略B（Close）的超额收益（ExcBps）全面低于策略A（VWAP）。Top-50 从 **+8.79** 掉到了 **+6.72**。
*   **解读**：这证明了我们之前的猜想——模型选出的股票中，有约 **2 bps** 的超额收益是“瞬时”的，在收盘前就已经回落了。
*   **结论**：如果你必须用收盘价一笔成交，你的 Alpha 天生就比用算法交易（VWAP）执行要薄 25% 左右。

#### 2. “收盘价训练”并没有让“收盘价离场”变好（表2 绩效对比）
*   **现象**：即便策略B在训练时已经知道了目标是收盘价，但在 2x 缓冲下，它的 NetBps（2.55）依然远逊于策略A（6.41）。
*   **核心原因（换手率）**：策略B的换手率显著高于策略A（Top-50: 58.5% vs 43.2%）。
*   **深度分析**：这说明**收盘价（Close）作为标签带有更高的随机噪声**。基于噪声大的标签训练出来的模型，其预测排名在日间极不稳定（Flickering），导致了更高的无谓换手成本。

#### 3. 极值区域的“反转”发现（表2 Top-20）
*   **惊人发现**：在 Top-20 极小池子里，策略B的 ExcBps（7.76）竟然反超了策略A（5.89）。
*   **解读**：这说明对于“最强信号”，寻找能够站稳收盘价的个股是可行的。但代价是 69% 的超高换手，直接把利润吞噬了。

---

### 二、 意见与观点

1.  **标签的“稳定性”优于“准确性”**：
    虽然老板要求时序思路，但实验证明，**VWAP 依然是比 Close 更好的训练基准**。因为它平滑了分笔成交的随机性，模型学到的特征更具有持续性，从而显著降低了交易成本（换手率低了 15% 左右）。

2.  **执行端才是生死线**：
    策略A在 2x 缓冲下有 **15.5% 的年化净收益**，而策略B只有 **6.2%**。这 9% 的差距不是模型带来的，而是**离场方式（VWAP vs Close）**带来的。
    *建议：* 哪怕在实盘中没有算法交易系统，也要尝试在 14:45 - 15:00 之间分批手动平仓，模拟 VWAP 效果，而不是等 15:00 最后一秒。

3.  **Alpha 的厚度问题**：
    目前策略B（Close版）的 NetBps 只有 2.55 bps。在真实交易中，如果滑点再大 1bp，这个策略就趋近于零利润。

---

### 三、 下一步行动建议：构建“鲁棒标签（Robust Label）”

既然 VWAP 标签太“乐观”，Close 标签太“随机”，我们接下来的目标是寻找一个**既能站稳收盘价、又具有时序平滑度**的训练目标。

#### 方案：引入“保守型混合标签（The Robust Target）”
我们将训练目标修改为：**$min(\text{Ret}_{\text{vwap}}, \text{Ret}_{\text{close}})$**。
*   **逻辑**：只有当一只票在 VWAP 维度和 Close 维度**都表现好**时，才给高分。这会强行滤掉那些“冲高回落”的脉冲股。

#### 详细代码实现：

你需要运行这个脚本生成新的“鲁棒标签”数据集，然后重新训练。

```python
"""
S11: 构建鲁棒标签训练集
目标：通过取 VWAP 收益和 Close 收益的交集，迫使模型学习稳定性 Alpha
"""

import pandas as pd
import numpy as np
from pathlib import Path
from loguru import logger

OUTPUT_DIR = Path("/nfs/volume-1593-1/peterzhenglinpeng/vwap-research/output")

def build_robust_label_dataset(year):
    # 1. 加载包含 close_ret 的预测数据（或者原始特征数据）
    # 确保你有 raw_ret (VWAP版) 和 close_ret (Close版)
    df = pd.read_pickle(OUTPUT_DIR / f"ts_lgbm_predictions_with_close.pkl")
    
    logger.info(f"处理 {year} 年鲁棒标签...")

    # 2. 计算超额收益
    mkt_vwap = df.groupby('date')['raw_ret'].transform('mean')
    mkt_close = df.groupby('date')['close_ret'].transform('mean')
    
    df['exc_vwap'] = df['raw_ret'] - mkt_vwap
    df['exc_close'] = df['close_ret'] - mkt_close

    # 3. 核心：构造鲁棒标签 (取保守值)
    # 如果 exc_vwap 是 20bps, exc_close 是 -10bps, 则标签是 -10bps
    # 这会告诉模型：这种冲高回落的票不是我们要的“正样本”
    df['robust_excess_ret'] = np.minimum(df['exc_vwap'], df['exc_close'])

    # 4. 也可以做波动率标准化
    # df['label'] = df['robust_excess_ret'] / df['hist_vol_20d']
    
    # 5. 保存供训练使用
    save_path = OUTPUT_DIR / f"df_ts_ready_robust_{year}.pkl"
    df.to_pickle(save_path)
    logger.success(f"鲁棒标签集已保存: {save_path}")

if __name__ == "__main__":
    for y in [2024, 2025]:
        build_robust_label_dataset(y)
```

### 四、 针对回测代码的 S6 改进建议（行业约束）

为了解决“爆炸日”和行业集中度风险，我建议你在 `ts_portfolio_backtest.py` 里的 `run_topn_backtest_turnover` 函数中加入以下逻辑：

```python
def select_top_n_with_sector_constraint(df_day, n=50, max_per_sector=3):
    """
    带行业约束的选股逻辑
    df_day: 当天所有股票的 pred_prob 和 SectorCode
    """
    selected = []
    sector_counts = {}
    
    # 按得分从高到低排序
    df_sorted = df_day.sort_values('pred_prob', ascending=False)
    
    for _, row in df_sorted.iterrows():
        sector = row['SectorCode'] # 假设你特征里有行业代码
        count = sector_counts.get(sector, 0)
        
        if count < max_per_sector:
            selected.append(row['SecuCode'])
            sector_counts[sector] = count + 1
        
        if len(selected) >= n:
            break
            
    return selected
```

### 总结：
你目前的实验已经证明了：**这个 Alpha 是真实的，但它是“脆弱”的。** 
通过 **鲁棒标签训练 (Target Engineering)** 和 **行业分散控制 (Risk Control)**，你应该能把策略B（Close离场版）的 NetBps 提升到 4-5 bps 左右，夏普比率提升到 1.0 以上。

**你想现在尝试一下这个“鲁棒标签（Robust Target）”训练，看看它的换手率是否会比纯 Close 标签更低吗？**